# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadfarhan2157-source/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [ ]:
# Rebuild the honest (grouped-split) model's queue for the full March slice
rf_final = RandomForestClassifier(n_estimators=200, random_state=1).fit(train_grouped[feature_cols], train_grouped["declined"])
all_scores = rf_final.predict_proba(data[feature_cols])[:, 1]

playbook_df = data.copy()
playbook_df["model_score"] = all_scores
playbook_df["stale_flag"] = ((playbook_df["content_age_days"] >= 180) & (playbook_df["impressions_march"] >= 500)).astype(int)
playbook_df["ctr_gap_flag"] = ((playbook_df["avg_position_march"] > 0) & (playbook_df["avg_position_march"] <= 20) &
                                 (playbook_df["ctr_march"] < 0.02) & (playbook_df["impressions_march"] >= 500)).astype(int)

def reason_code(row):
    if row["stale_flag"] and row["ctr_gap_flag"]:
        return "stale_and_low_ctr"
    elif row["stale_flag"]:
        return "stale_visible_page"
    elif row["ctr_gap_flag"]:
        return "low_ctr_visible_page"
    return "no_flag"

playbook_df["reason_code"] = playbook_df.apply(reason_code, axis=1)
playbook_df["action"] = playbook_df["reason_code"].apply(lambda r: "refresh" if r != "no_flag" else "monitor")
playbook_df = playbook_df.sort_values("model_score", ascending=False)

playbook_df.head(20)[["content_hash_id", "model_score", "reason_code", "action"]]

The queue ranks pages the model flags as likely to decline, at the Precision@50 measured in Section 3 of w06. Reason codes (stale_and_low_ctr, stale_visible_page, low_ctr_visible_page) explain why each page ranked where it did — a reviewer should be able to see the reason and agree or disagree with it, not just trust the number.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Intended use: decision-support for a content reviewer with limited weekly capacity — a starting point for "look at this first," not an automatic action trigger.
Who uses this: a content lead prioritizing a review queue, not an automated publishing pipeline.
Where it stops being valid: outside month=2026-03–like conditions (a different season, a client with under 90 days of history, or the sealed final month this was never validated against). Per the claim ladder, this is cross-sectional-adjacent (one client-holdout split, one time window) — it does not support "refreshing this page will cause a recovery." The honest form: "these pages look worth reviewing first, because [reason code]."
Selection bias note: the training label itself (declined) reflects which pages organically saw fewer April clicks — it says nothing about pages a human already chose to refresh in that window, so the model hasn't seen intervention effects at all.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

A person must check, before acting on any row: whether the page is intentionally static (a policy/reference page where "staleness" is by design), whether flagged impressions reflect a one-off spike rather than steady demand, and whether the reason code actually matches what they see when they open the page.

What should NOT be automated:

Auto-publishing refreshed content based on the score alone.
Auto-deprioritizing "no_flag" pages — absence of a flag isn't evidence of health, just absence of this model's specific signals.
Treating model_score as a ranking of page quality — it's a ranking of predicted decline risk under one label definition, not an overall quality judgment.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

What would tell me the recommendations are going stale: a large gap opening between the w06 grouped-split Precision@50 and the queue's real-world hit rate once reviewers report back; a new client entering the slice with less than 90 days of history (per the lane guide's client-history caveat) skewing the feature distributions; or the base-rate of decline itself shifting meaningfully month over month, which would mean the model's implicit assumptions about "normal" no longer hold.
Retrain trigger: re-run monthly on the newest completed (non-sealed) month, and re-check the random-vs-grouped split gap each time — if that gap widens, something about client-level memorization is creeping back in.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
import os, json

os.makedirs("../outputs", exist_ok=True)
os.makedirs("../figures", exist_ok=True)

playbook_df.to_csv("../outputs/baseline_action_score.csv", index=False)  # stays out of git by design

metrics_receipt = {
    "month_used": "2026-03",
    "label_window": "2026-04",
    "base_rate": float(base_rate),
    "grouped_split_roc_auc": float(roc_auc_score(test_grouped["declined"], grouped_scores)),
    "grouped_split_precision_at_50": float(precision_at_k(test_grouped["declined"].reset_index(drop=True), pd.Series(grouped_scores))),
    "naive_split_roc_auc_for_comparison": float(roc_auc_score(test_naive["declined"], naive_scores)),
    "leak_experiment_auc_with_leak": float(roc_auc_score(test_l["declined"], leaky_scores)),
}

with open("../outputs/w07_metrics_receipt.json", "w") as f:
    json.dump(metrics_receipt, f, indent=2)

print("Exported queue CSV (not committed) and metrics receipt (committed) to work/outputs/")
print(metrics_receipt)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.